# ☁ Cloud Firewall Rule Analyzer
**Python project, simulated cloud, runs in Google Colab**

It reads firewall rules of a cloud (like AWS Security Groups), checks if they are correct, finds the risky rules, gives a risk score, links each problem to a security standard, and makes a report with charts.

**Flow:** `firewall_rules.json` → Load → Validate → Analyze → Score → Compliance map → Report (table + CSV + charts)

| Module | What it does |
|---|---|
| 1 | Create the data, load it and check it |
| 2 | Analyze the rules (security checks) |
| 3 | Risk scoring |
| 4 | Compliance mapping |
| 5 | Report and charts |
| 6 | Testing |

**How to run:** click **Runtime → Run all**. No upload or API key is needed.

## Section 1: Install and import libraries

In [ ]:
!pip install -q tabulate
import json
import ipaddress
import pandas as pd
import matplotlib.pyplot as plt
from tabulate import tabulate
from matplotlib.ticker import MaxNLocator
print("Libraries ready")

## Section 2: Create the dataset (simulated security-group rules)
We cannot use a real cloud account, so we make our own sample rules. Each rule has a `rule_id`, `group` (security group), `protocol`, `port` and `source` (who can connect).

The data has safe, risky, duplicate and invalid rules on purpose, so that every check can be shown.

In [ ]:
rules = [
    {"rule_id": 1,  "group": "web-sg",   "protocol": "tcp", "port": "80",    "source": "0.0.0.0/0"},        # public web
    {"rule_id": 2,  "group": "web-sg",   "protocol": "tcp", "port": "22",    "source": "0.0.0.0/0"},        # SSH open to world
    {"rule_id": 3,  "group": "db-sg",    "protocol": "tcp", "port": "3306",  "source": "0.0.0.0/0"},        # MySQL open to world
    {"rule_id": 4,  "group": "web-sg",   "protocol": "tcp", "port": "443",   "source": "0.0.0.0/0"},        # public HTTPS
    {"rule_id": 5,  "group": "admin-sg", "protocol": "tcp", "port": "22",    "source": "203.0.113.5/32"},   # SSH from one trusted IP
    {"rule_id": 6,  "group": "app-sg",   "protocol": "all", "port": "all",   "source": "0.0.0.0/0"},        # allow everything
    {"rule_id": 7,  "group": "db-sg",    "protocol": "tcp", "port": "5432",  "source": "10.0.0.0/8"},       # very wide range
    {"rule_id": 8,  "group": "web-sg",   "protocol": "tcp", "port": "80",    "source": "0.0.0.0/0"},        # duplicate of rule 1
    {"rule_id": 9,  "group": "app-sg",   "protocol": "tcp", "port": "8080",  "source": "999.1.1.1/24"},     # invalid CIDR
    {"rule_id": 10, "group": "admin-sg", "protocol": "tcp", "port": "3389",  "source": "0.0.0.0/0"},        # RDP open to world
    {"rule_id": 11, "group": "app-sg",   "protocol": "tcp", "port": "70000", "source": "192.168.1.0/24"},   # invalid port
    {"rule_id": 12, "group": "db-sg",    "protocol": "tcp", "port": "1433",  "source": "198.51.100.0/24"},  # restricted SQL Server
    {"rule_id": 13, "group": "app-sg",   "protocol": "tcp", "port": "8080",  "source": "192.168.1.0/24"},   # safe internal app
]

with open("firewall_rules.json", "w") as f:
    json.dump(rules, f, indent=2)
print(f"firewall_rules.json created with {len(rules)} rules")

## Section 3: Module 1 - Data loading and validation
- The JSON file is loaded into a pandas DataFrame.
- `ipaddress.ip_network()` checks that the source is a proper CIDR. For example `999.1.1.1/24` is wrong.
- The port must be from 1 to 65535 or `"all"`.
- The protocol must be `tcp`, `udp`, `icmp` or `all`.

**Security mechanism: input validation** - wrong data is marked as invalid, we do not trust it blindly.

In [ ]:
VALID_PROTOCOLS = {"tcp", "udp", "icmp", "all"}
REQUIRED_FIELDS = ["rule_id", "group", "protocol", "port", "source"]

def is_valid_cidr(cidr):
    try:
        ipaddress.ip_network(cidr, strict=False)
        return True
    except ValueError:
        return False

def is_valid_port(port):
    if port == "all":
        return True
    return str(port).isdigit() and 1 <= int(port) <= 65535

def validate_rule(rule):
    """Return a list of error messages (empty list = rule is valid)."""
    errors = []
    if not is_valid_cidr(rule["source"]):
        errors.append(f"Invalid CIDR {rule['source']}")
    if not is_valid_port(rule["port"]):
        errors.append(f"Invalid port {rule['port']}")
    if rule["protocol"] not in VALID_PROTOCOLS:
        errors.append(f"Invalid protocol {rule['protocol']}")
    return errors

def build_dataframe(rule_list):
    df = pd.DataFrame(rule_list)
    df["port"] = df["port"].astype(str).str.lower()
    df["protocol"] = df["protocol"].astype(str).str.lower()
    df["errors"] = [validate_rule(r) for r in df.to_dict("records")]
    df["valid"] = df["errors"].apply(len) == 0
    return df

def load_rules(path):
    with open(path) as f:
        rule_list = json.load(f)
    for i, rule in enumerate(rule_list, start=1):
        missing = [k for k in REQUIRED_FIELDS if k not in rule]
        if missing:
            raise ValueError(f"Rule number {i} is missing: {', '.join(missing)}")
    return build_dataframe(rule_list)

df = load_rules("firewall_rules.json")
print(f"Loaded {len(df)} rules | valid: {df['valid'].sum()} | invalid: {(~df['valid']).sum()}\n")
print(tabulate(df[["rule_id", "group", "protocol", "port", "source", "valid"]],
               headers="keys", tablefmt="grid", showindex=False))

## Section 4: Module 2 - Rule analysis (the security checks)

| # | Check | How it is checked | Severity |
|---|---|---|---|
| 1 | Allow-all rule | protocol = all and port = all | High |
| 2 | Risky port open to internet | source = 0.0.0.0/0 and port is 22, 3389, 3306, 5432 or 1433 | High |
| 3 | Other port open to internet | source = 0.0.0.0/0 | Medium |
| 4 | Wide CIDR range | prefix is /16 or bigger | Medium |
| 5 | Duplicate rule | same group, protocol, port and source | Low |
| 6 | Safe | none of the above problems | Safe |

**Security mechanism: least privilege** - only the traffic which is really needed should be allowed, and only from a small source range.

In [ ]:
RISKY_PORTS = {"22": "SSH", "3389": "RDP", "3306": "MySQL",
               "5432": "PostgreSQL", "1433": "SQL Server"}
ADMIN_PORTS = {"22", "3389"}   # remote administration ports

def make_finding(severity, finding, recommendation, compliance):
    return {"severity": severity, "finding": finding,
            "recommendation": recommendation, "compliance": compliance}

def is_open_to_world(source):
    """True for 0.0.0.0/0 (and ::/0). A /0 prefix means every address."""
    return ipaddress.ip_network(source, strict=False).prefixlen == 0

def analyze_rule(rule):
    """Run checks 1-4 on ONE valid rule. Returns a list of findings."""
    findings = []

    # Check 1: allow-all
    if rule["protocol"] == "all" and rule["port"] == "all":
        findings.append(make_finding("High", "Allow-all rule",
                                     "Restrict to required protocols and ports", "allow_all"))

    if is_open_to_world(rule["source"]):
        # Check 2: risky port open to the internet
        if rule["port"] in RISKY_PORTS:
            kind = "admin" if rule["port"] in ADMIN_PORTS else "database"
            findings.append(make_finding("High", f"{RISKY_PORTS[rule['port']]} open to the internet",
                                         "Limit source to a trusted IP or VPN", kind))
        # Check 3: any other specific port open to the internet
        elif rule["port"] != "all":
            findings.append(make_finding("Medium", "Port open to the internet",
                                         "Confirm this port needs public access", "public_port"))
    # Check 4: wide CIDR range (open-to-world is already covered above)
    elif ipaddress.ip_network(rule["source"], strict=False).prefixlen <= 16:
        findings.append(make_finding("Medium", "Wide CIDR range",
                                     "Narrow the source range to specific subnets", "wide_cidr"))
    return findings

def find_duplicates(rule_df):
    """Check 5: return the rule_ids that repeat an earlier rule."""
    seen, duplicates = set(), set()
    for r in rule_df.to_dict("records"):
        key = (r["group"], r["protocol"], r["port"], r["source"])
        if key in seen:
            duplicates.add(r["rule_id"])
        else:
            seen.add(key)
    return duplicates

# Quick demo on single rules
print(analyze_rule({"protocol": "tcp", "port": "22", "source": "0.0.0.0/0"}))
print(analyze_rule({"protocol": "tcp", "port": "22", "source": "203.0.113.5/32"}))

## Section 5: Module 3 - Risk scoring
- Score of one finding: High = 3, Medium = 2, Low = 1, Safe = 0, Invalid = 0
- Score of a security group = sum of its finding scores
- Score of the whole environment = sum of all group scores
- Group rating: 6 or more = **HIGH RISK**, 3 or more = **MEDIUM RISK**, else **LOW RISK**

In [ ]:
SEVERITY_SCORE = {"High": 3, "Medium": 2, "Low": 1, "Safe": 0, "Invalid": 0}
SEVERITY_ORDER = {"High": 0, "Medium": 1, "Low": 2, "Invalid": 3, "Safe": 4}

def analyze_all(rule_df):
    """Analyze every rule and return one report row per finding."""
    duplicates = find_duplicates(rule_df[rule_df["valid"]])
    rows = []
    for r in rule_df.to_dict("records"):
        if not r["valid"]:
            findings = [make_finding("Invalid", "; ".join(r["errors"]), "Fix the rule data", "invalid")]
        else:
            findings = analyze_rule(r)
            if r["rule_id"] in duplicates:
                findings.append(make_finding("Low", "Duplicate rule", "Remove the repeated rule", "duplicate"))
            if not findings:
                findings = [make_finding("Safe", "No issue found", "None", "safe")]
        for f in findings:
            rows.append({"rule_id": r["rule_id"], "group": r["group"], "protocol": r["protocol"],
                         "port": r["port"], "source": r["source"], **f,
                         "score": SEVERITY_SCORE[f["severity"]]})
    report = pd.DataFrame(rows)
    report["_order"] = report["severity"].map(SEVERITY_ORDER)
    return report.sort_values(["_order", "rule_id"]).drop(columns="_order").reset_index(drop=True)

def rate(score):
    return "HIGH RISK" if score >= 6 else "MEDIUM RISK" if score >= 3 else "LOW RISK"

def score_groups(report):
    groups = report.groupby("group")["score"].sum().reset_index(name="risk_score")
    groups["rating"] = groups["risk_score"].apply(rate)
    return groups.sort_values("risk_score", ascending=False).reset_index(drop=True)

def worst_per_rule(report):
    """One row per rule: its most severe finding."""
    tmp = report.assign(_o=report["severity"].map(SEVERITY_ORDER)).sort_values(["rule_id", "_o"])
    return tmp.drop_duplicates("rule_id").drop(columns="_o")

report = analyze_all(df)
group_scores = score_groups(report)
env_score = int(report["score"].sum())

print("Risk score per security group")
print(tabulate(group_scores, headers=["Group", "Score", "Rating"], tablefmt="grid", showindex=False))
print(f"\nOverall environment risk score: {env_score}")

## Section 6: Module 4 - Compliance mapping
Every finding is linked to a security standard. This shows **why** the rule is a problem and not only that it is risky.

*This mapping is only a simple example for learning. The exact control numbers can be different in different versions of the standards.*

In [ ]:
COMPLIANCE_MAP = {
    "admin":       "CIS AWS Foundations (no SG ingress from 0.0.0.0/0 to remote admin ports); NIST SP 800-53 SC-7",
    "database":    "PCI DSS Req. 1 (restrict access to data systems); NIST SP 800-53 SC-7",
    "allow_all":   "PCI DSS Req. 1 (allow only necessary traffic); NIST SP 800-53 SC-7",
    "public_port": "NIST SP 800-53 SC-7 (boundary protection); least privilege",
    "wide_cidr":   "Least privilege; PCI DSS Req. 1",
    "duplicate":   "Configuration hygiene / change management",
    "invalid":     "Input validation / data integrity",
    "safe":        "Compliant with least privilege",
}

report["compliance_ref"] = report["compliance"].map(COMPLIANCE_MAP)

issues = report[report["severity"] != "Safe"]
print(tabulate(issues[["rule_id", "severity", "finding", "compliance_ref"]],
               headers=["Rule", "Severity", "Finding", "Compliance reference"],
               tablefmt="grid", showindex=False, maxcolwidths=[None, None, 28, 60]))

## Section 7: Module 5 - Reporting (results table and summary)
The final table shows every rule with its severity, finding and recommendation. The summary line gives the total count.

In [ ]:
def print_report(report):
    table = report[["rule_id", "group", "port", "source", "severity", "finding", "recommendation"]]
    print(tabulate(table, headers=["Rule", "Group", "Port", "Source", "Severity", "Finding", "Recommendation"],
                   tablefmt="grid", showindex=False))

def print_summary(report):
    worst = worst_per_rule(report)
    n = {s: int((worst["severity"] == s).sum()) for s in ["High", "Medium", "Low", "Invalid", "Safe"]}
    risky = n["High"] + n["Medium"] + n["Low"]
    print(f"SUMMARY: {len(worst)} rules analyzed, {risky} risky "
          f"({n['High']} High, {n['Medium']} Medium, {n['Low']} Low), "
          f"{n['Invalid']} invalid, {n['Safe']} safe")

print_report(report)
print()
print_summary(report)

### Charts
The first chart shows the number of findings by severity. The second chart shows the risky rules in each security group.

In [ ]:
colors = {"High": "#d9534f", "Medium": "#f0ad4e", "Low": "#5bc0de", "Invalid": "#777777", "Safe": "#5cb85c"}

def make_charts(report, path="charts.png"):
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    # Chart 1: findings by severity
    counts = report["severity"].value_counts().reindex(["High", "Medium", "Low", "Invalid", "Safe"], fill_value=0)
    axes[0].bar(counts.index, counts.values, color=[colors[s] for s in counts.index])
    axes[0].set_title("Findings by Severity")
    axes[0].set_ylabel("Number of findings")
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 0.05, str(v), ha="center")

    # Chart 2: risky rules per security group (stacked by severity)
    worst = worst_per_rule(report)
    risky = worst[worst["severity"].isin(["High", "Medium", "Low"])]
    if risky.empty:
        axes[1].text(0.5, 0.5, "No risky rules", ha="center", va="center")
        axes[1].set_axis_off()
    else:
        per_group = (pd.crosstab(risky["group"], risky["severity"])
                     .reindex(columns=["High", "Medium", "Low"], fill_value=0))
        per_group.plot(kind="bar", stacked=True, ax=axes[1], color=[colors[c] for c in per_group.columns])
        axes[1].set_title("Risky Rules per Security Group")
        axes[1].set_ylabel("Number of risky rules")
        axes[1].set_xlabel("")
        axes[1].tick_params(axis="x", rotation=0)

    for ax in axes:
        ax.yaxis.set_major_locator(MaxNLocator(integer=True))   # whole numbers only
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()

make_charts(report)

## Section 8: Save the CSV report
The report is saved as `report.csv` and the chart is saved as `charts.png`. In Colab they are also downloaded to your computer.

In [ ]:
export = report[["rule_id", "group", "protocol", "port", "source", "severity",
                 "finding", "recommendation", "compliance_ref", "score"]]
export.to_csv("report.csv", index=False)
print("report.csv saved:", len(export), "rows")

# Download to your computer (works only inside Google Colab)
try:
    from google.colab import files
    files.download("report.csv")
    files.download("charts.png")
except ImportError:
    print("Not running in Colab - find report.csv and charts.png in the current folder.")

## Section 9: Module 6 - Testing
Each test gives one known rule to the same validation and analysis code, and then compares the result with the expected severity. The last two checks test the sample data and the scoring.

In [ ]:
def run_case(rule_list):
    """Run rules through the full pipeline and return the set of severities found."""
    return set(analyze_all(build_dataframe(rule_list))["severity"])

def r(i, port, source, protocol="tcp", group="test-sg"):
    return {"rule_id": i, "group": group, "protocol": protocol, "port": port, "source": source}

tests = [
    ("SSH from 0.0.0.0/0",       [r(1, "22", "0.0.0.0/0")],                                    "High"),
    ("SSH from 203.0.113.5/32",  [r(1, "22", "203.0.113.5/32")],                               "Safe"),
    ("Allow-all rule",           [r(1, "all", "0.0.0.0/0", protocol="all")],                   "High"),
    ("Port 80 from 0.0.0.0/0",   [r(1, "80", "0.0.0.0/0")],                                    "Medium"),
    ("Invalid CIDR 999.1.1.1/24",[r(1, "80", "999.1.1.1/24")],                                 "Invalid"),
    ("Invalid port 70000",       [r(1, "70000", "192.168.1.0/24")],                            "Invalid"),
    ("Invalid protocol",         [r(1, "80", "10.1.1.0/24", protocol="http")],                 "Invalid"),
    ("Same rule twice",          [r(1, "22", "203.0.113.5/32"), r(2, "22", "203.0.113.5/32")], "Low"),
    ("Wide CIDR 10.0.0.0/8",     [r(1, "443", "10.0.0.0/8")],                                  "Medium"),
    ("RDP from IPv6 ::/0",       [r(1, "3389", "::/0")],                                       "High"),
]

results = []
for name, rule_list, expected in tests:
    found = run_case(rule_list)
    results.append([name, expected, ", ".join(sorted(found)), "PASS" if expected in found else "FAIL"])

print(tabulate(results, headers=["Test case", "Expected", "Actual", "Result"], tablefmt="grid"))
assert all(row[3] == "PASS" for row in results), "Some tests failed"
print(f"\nAll {len(results)} tests passed")

# Check the sample data gives the expected numbers
assert env_score == 21, "Environment score should be 21"
assert dict(zip(group_scores["group"], group_scores["risk_score"])) == \
       {"web-sg": 10, "db-sg": 5, "app-sg": 3, "admin-sg": 3}, "Group scores are wrong"
assert (rate(6), rate(3), rate(2)) == ("HIGH RISK", "MEDIUM RISK", "LOW RISK"), "Rating levels are wrong"
print("Sample data and scoring checks passed")

## Section 10 (optional): Analyze your own rules file
Set `USE_OWN_FILE = True` and run the cell. Colab will ask you to choose a JSON file. It must be a list of rules with the fields `rule_id`, `group`, `protocol`, `port` and `source` (same as `firewall_rules.json` above).

In [ ]:
USE_OWN_FILE = False   # change to True to upload your own JSON file

if USE_OWN_FILE:
    from google.colab import files
    uploaded = files.upload()
    my_df = load_rules(list(uploaded)[0])
    my_report = analyze_all(my_df)
    my_report["compliance_ref"] = my_report["compliance"].map(COMPLIANCE_MAP)

    print_report(my_report)
    print()
    print_summary(my_report)
    print()
    print(tabulate(score_groups(my_report), headers=["Group", "Score", "Rating"], tablefmt="grid", showindex=False))
    make_charts(my_report, "my_charts.png")
else:
    print("Skipped. Set USE_OWN_FILE = True to analyze your own file.")

## Section 11: Conclusion
- The analyzer loads the firewall rules and checks them, so wrong CIDR, ports and protocols are found.
- It runs security checks to find allow-all rules, risky ports open to the internet, wide ranges and duplicates.
- Each finding gets a severity, a risk score and a recommendation. It is also linked to a standard (CIS, PCI DSS, NIST).
- Output: results table, `report.csv` and charts.

**Future work:** connect to real AWS using `boto3`, read CSV files, and make an HTML/PDF report.